# 人工智能与算子基础
本节是Ascend C算子开发的基础章节，将会按照如下结构，带你快速了解AI及算子。
- 人工智能的发展历程
- 什么是算子

---

## **1. 人工智能的发展历程**
算子实际上是人工智能计算的“底层积木”，所以在了解算子之前，首先要对人工智能有一个基础的认知。

### 1.1 人工智能（AI）缘起：从图灵测试到达特茅斯会议
早在 1950 年，现代计算机科学之父阿兰・图灵在《计算机器与智能》中提出了图灵测试，首次将“机器是否具备人类智能”的探讨从哲学思辨推向科学研究维度。这一测试的核心逻辑是：测试者与被测试者（一人一机）相互隔离，通过键盘等装置随机提问；经过多次测试后，若机器能让平均每位测试者做出超过 30% 的误判，即被认为具备人类智能。

<img src="./images/turing_test.png" alt="turing_test"  width="700px" >

值得一提的是，俄罗斯团队开发的聊天机器人尤金・古斯特，曾号称“史上首个通过图灵测试的智能体”（测试通过率33%）。其核心思路是巧妙利用规则漏洞 —— 冒充来自乌克兰、英语非母语的13岁小孩，在5分钟的测试时长内降低评委的判断标准，从而达成“骗过人类”的效果。

真正标志着人工智能成为独立研究学科的里程碑，是1956年的达特茅斯会议。由约翰・麦卡锡等学者首次明确提出“人工智能（Artificial Intelligence）”的概念，定义其核心目标是让机器模仿人类的感知、认知、决策与执行能力，自此开启了人工智能的系统性研究历程。
  
### 1.2 人工智能（AI）发展：从“感知理解世界” 向“生成创造世界”延展
下图完整梳理了人工智能自诞生以来的核心发展阶段与关键里程碑，清晰展现了AI从 “分析数据” 到 “创造内容” 的演进路径。

<img src="./images/artificial_intelligence_development.png" alt="artificial_intelligence_development"  width="700px" >

整体来看，AI的发展脉络呈现出 “从专用智能到通用智能” 的核心趋势：早期AI只能解决单一领域的特定问题（如围棋AI、语音识别），而如今的大模型具备跨领域、多模态的通用能力，能同时处理文本、图像、语音等多种数据类型，逐步逼近“通用人工智能”的目标。

### 1.3 人工智能（AI）应用加速：从局部探索走向千行百业 
人工智能正在行业内不断渗透，从大众熟悉的智能分拣、人脸识别，到工业质检、医疗影像分析等非大众场景，都在通过AI提升效率、优化流程。
  
<img src="./images/ai_industry_application_acceleration.png" alt="ai_industry_application_acceleration"  width="700px" >

---

## **2. 什么是算子**
在建立人工智能的基础认知后，我们将从算子含义和基本概念两方面，拆解算子的定义与核心构成。

### 2.1 算子的含义
人工智能的核心是模型算法，而算法本质是由一个个基础计算单元构成的，这些计算单元被称为**算子（Operator，简称OP）**。在网络模型中，算子对应具体层或节点的计算逻辑，模型通常由一个或多个算子组合拼接而成。

如下图所示，基于PyTorch典型模型导出后，在Netron中可视化呈现的模型结构里，图中的Conv模块就是典型的卷积算子，而这类算子正是构成AI模型的基础计算单元。

<img src="./images/pytorch_net.png" alt="pytorch_net"  width="700px" >

在数学领域一个函数空间到函数空间上的映射O：X→Y，都称为算子。广义的讲，对任何函数进行某一项操作都可以认为是一个算子，比如微分算子，不定积分算子等。常见算子如tanh、ReLU、sigmoid等。

<img src="./images/common_operator_images.png" alt="common_operator_images"  width="700px" >

### 2.2 算子核心基本概念
理解算子的使用与设计，需掌握其关联的六大核心概念，包括算子自身属性、数据容器及容器的关键描述维度，具体如下。

#### 2.2.1 算子名称（Name）与算子类型（Type）  
算子名称是网络中单个算子的**唯一标识**，同一网络内不可重复，用于精准区分不同算子。

算子类型决定了算子的**核心实现逻辑**，相同类型的算子执行完全一致的计算逻辑，一个网络中可存在多个同类型算子。
  
如下图所示，Conv1、Pool1、Conv2是该网络中的算子名称，其中Conv1和Conv2的算子类型均为 Convolution，代表二者执行的都是卷积计算。

<img src="./images/basic_concepts_of_operators_1.png" alt="basic_concepts_of_operators_1"  width="700px" >

如下是torch网络构建的代码片段，可以明确看出。Conv2d、MaxPool2d、ReLU等是算子类型，而conv1、conv2、pool1等是算子名称。
```
class SimpleConvNet(nn.Module):
    def __init__(self):
        super().__init__()
        # 1. 自定义算子名称：为2个同类型（Convolution）卷积算子分别命名conv1、conv2
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)  # 卷积算子1
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1) # 卷积算子2
        # 2. 自定义算子名称：池化算子（MaxPool2d类型）命名pool1，激活算子（ReLU类型）命名relu1
        self.pool1 = nn.cc(kernel_size=2, stride=2)  # 池化算子
        self.relu1 = nn.ReLU()  # 激活算子
        # 3. 未显式命名（PyTorch自动生成唯一名称）：新增一个ReLU算子
        self.add_module("relu2", nn.ReLU())  # 等价于直接self.relu2 = nn.ReLU()

    def forward(self, x):
        x = self.relu1(self.conv1(x))
        x = self.pool1(x)
        x = self.relu2(self.conv2(x))
        return x
```

#### 2.2.2 数据容器（Tensor）  
算子在网络中执行计算，需要输入数据作为计算基础，计算完成后也会输出对应结果，**张量（Tensor） 就是专门承载算子输入、输出数据的容器**。

<img src="./images/basic_concepts_of_operators_2.png" alt="basic_concepts_of_operators_2"  width="280px" >
  
Tensor（张量）是一种多维数组形式的数据容器，可理解为标量、向量、矩阵的高维拓展，专门用于高效存储和处理各类批量多维数据，是AI计算、并行计算中数据传递与运算的标准载体。传统编程中常以标量、向量、矩阵表示固定维度的数据，而算子开发中需支持任意维度的数处理，因此采用张量这一通用概念来统一表征，相关实现可参考如下代码。

In [ ]:
import numpy as np

# 1. 创建不同维度的Tensor（NumPy的ndarray即为实操中的Tensor）
scalar = np.array(5)  # 0维Tensor（标量）
vec = np.array([1, 2, 3])  # 1维Tensor（向量）
mat = np.array([[1, 2], [3, 4]])  # 2维Tensor（矩阵）
tensor_3d = np.array([[[1,2], [3,4]], [[5,6], [7,8]]])  # 3维Tensor

# 打印Tensor及基本信息
print("【0维Tensor（标量）】：", scalar, "，shape：", scalar.shape)
print("【1维Tensor（向量）】：", vec, "，shape：", vec.shape)
print("【2维Tensor（矩阵）】：\n", mat, "，shape：", mat.shape)
print("【3维Tensor】：\n", tensor_3d, "，shape：", tensor_3d.shape)

# 2. 体验“算子是基础计算单元”：用「加法算子」对Tensor做计算
add_op = lambda x, y: x + y  # 定义加法算子（lambda简化函数）

# 加法算子对不同Tensor的计算（同维度才能计算，体现算子的计算逻辑）
res1 = add_op(vec, [4,5,6])  # 1维Tensor相加
res2 = add_op(mat, [[5,6], [7,8]])  # 2维Tensor相加
print("\n【加法算子计算结果】")
print("1维Tensor相加：", res1)
print("2维Tensor相加：\n", res2)

根据代码，尝试以下操作理解Tensor的概念。
- 运行代码，理解 “不同维度的Tensor是算子的计算数据”；
- 尝试用减法/乘法定义新算子（如mul_op = lambda x,y: x*y），对Tensor做计算；
- 故意让不同维度Tensor相加（如add_op(vec, mat)），看报错，理解算子的计算约束。

#### 2.2.3 张量描述符（TensorDesc）  
Tensor 作为数据容器，存放算子计算所需的实际输入/输出数值，相当于一杯装了水的杯子.

张量描述符（TensorDesc） 不存储任何实际数据，仅用于记录描述Tensor的元信息，相当于杯子上的标签。硬件层会先通过该标签读懂数据属性，再高效处理数据，这是适配异构计算、提升数据处理效率的经典设计。

张量描述符（TensorDesc）的核心数据结构包含以下属性：
  
<table style="float: left; border-collapse: collapse; margin: 0 10px 10px 0; font-size: 14px;">
<tr style="background-color:#f0f0f0">
  <td align="center"><strong>属性</strong></td>
  <td align="center"><strong>说明</strong></td>
</tr>
<tr>
  <td align="left">名称（name）</td>
  <td align="left">Tensor的唯一标识，用于索引和区分不同Tensor</td>
</tr>
<tr>
  <td align="left">形状（shape）</td>
  <td align="left">描述Tensor的维度大小，格式为(i1,i2,…,in)，如(10,)、(2,3,4)，i为正整数</td>
</tr>
<tr>
  <td align="left">数据类型（dtype）</td>
  <td align="left">指定Tensor存储的数据类型，如float16/32、int8/32、bool等</td>
</tr>
<tr>
  <td align="left">数据排布格式（format）</td>
  <td align="left">定义数据的物理存储排布方式，如NCHW、NHWC、ND等</td>
</tr>
</table>
<div style="clear: both;"></div>

值得一提的是，PyTorch和NumPy中，均将TensorDesc的描述能力直接内置在Tensor对象中，而CANN算子开发里，Tensor与TensorDesc是相互分离的设计。  
这并非意味着TensorDesc是CANN的独有概念，二者只是设计逻辑不同，核心的“数据本体+属性描述”本质完全一致。  
此外，PyTorch 2.0 及以上版本也提供了torch.TensorSpec接口，可显式定义Tensor的规格，包含shape、dtype、device等核心信息，功能与CANN的TensorDesc完全等价。

In [ ]:
import numpy as np

# 定义函数：提取Tensor的元信息，模拟CANN的TensorDesc
def get_tensor_desc(tensor, name):
    """
    模拟TensorDesc，返回Tensor的核心描述信息
    :param tensor: 输入的Tensor（np.ndarray）
    :param name: Tensor的唯一名称（name属性）
    :return: 字典形式的TensorDesc
    """
    tensor_desc = {
        "name": name,  # 唯一标识
        "shape": tensor.shape,  # 形状
        "dtype": tensor.dtype,  # 数据类型
        "format": "ND" if len(tensor.shape)>=1 else "SCALAR"  # 排布格式，简化为ND/标量
    }
    return tensor_desc

# 创建模拟图片的4D Tensor：shape=(4,20,20,3)（4张20*20的RGB图片）
img_tensor = np.random.rand(4,20,20,3).astype(np.float32)  # dtype指定为float32
# 提取TensorDesc元信息
img_desc = get_tensor_desc(img_tensor, "rgb_image_tensor")

# 打印TensorDesc，对应上方表格
print("【张量描述符（TensorDesc）】")
for k, v in img_desc.items():
    print(f"{k}：{v}")
# 打印Tensor的实际数据（前2个像素）
print("\n【Tensor实际数据（前2个像素）】：\n", img_tensor[0,0,0:2,:])

根据代码，尝试以下操作理解TensorDesc。
- 运行代码，看TensorDesc的4个核心属性如何与Tensor对应，理解“标签（Desc）+ 数据（Tensor）”的关系。
- 修改img_tensor = np.random.rand(4,20,20,3).astype(np.float32)的dtype为np.int8/np.bool_，看TensorDesc的dtype变化。

#### 2.2.4 形状（shape）  
Tensor的形状，以(D0, D1, … ,Dn-1)的形式表示，D0到Dn是任意的正整数。如形状(3,4)表示第一维有3个元素，第二维有4个元素，是一个3行4列的矩阵数组，以下是一些典型Tensor与对应的shape表示。

<table style="float: left; border-collapse: collapse; margin: 0 10px 10px 0; font-size: 14px;">
<tr style="background-color:#f0f0f0">
  <td align="center"><strong>张量</strong></td>
  <td align="center"><strong>形状</strong></td>
</tr>
<tr>
  <td align="center">1</td>
  <td align="left">(0,)</td>
</tr>
<tr>
  <td align="center">[1,2,3]</td>
  <td align="left">(3,)</td>
</tr>
<tr>
  <td align="center">[[1,2],[3,4]]</td>
  <td align="left">(2,2)</td>
</tr>
<tr>
  <td align="center">[[[1,2],[3,4]], [[5,6],[7,8]]]</td>
  <td align="left">(2,2,2)</td>
</tr>
</table>
<div style="clear: both;"></div>

shape=(4, 20, 20, 3)的物理含义：假设有4张照片，即shape里4的含义，每张照片的宽和高都是20，也就是20*20=400个像素，每个像素点都由红/绿/蓝3色组成，即shape里面3的含义。体现在编程上，可以简单把shape理解为操作Tensor的各层循环，比如对shape=(4, 20, 20, 3)的A tensor进行操作，循环语句与示意图如下。

<img src="./images/detailed_explanation_of_shape.png" alt="detailed_explanation_of_shape"  width="700px" >

In [ ]:
import numpy as np

# 创建shape=(4,20,20,3)的Tensor：4张20*20的RGB图片，像素值[0,255]
img_np = np.random.randint(0, 256, size=(4,20,20,3), dtype=np.uint8)

# 多层循环遍历（对应shape的4个维度）
print("【遍历第1张图的前2行2列像素】")
for i in range(1):  # 只遍历第1张图（i=0）
    for j in range(2):  # 前2行（高）
        for p in range(2):  # 前2列（宽）
            for q in range(3):  # RGB3通道
                print(f"第{i+1}张图-({j},{p})像素-{['R','G','B'][q]}通道：{img_np[i,j,p,q]}")

根据代码，尝试以下操作理解shape。
- 运行代码，看shape的每个维度如何对应循环层数，理解“shape = 循环层级”的概念。
- 将range(1)改为range(4)，遍历所有4张图片的像素。

#### 2.2.5 数据排布格式（format）  
卷积神经网络的特征图（Feature Map）通常用四维数组保存，即4D格式：
- N：Batch数量，例如图像的数目。
- H：Height，特征图高度，即垂直高度方向的像素个数。
- W：Width，特征图宽度，即水平宽度方向的像素个数。
- C：Channels，特征图通道，例如彩色RGB图像的Channels为3。

**由于数据只能线性存储，因此这四个维度有对应的顺序**。不同深度学习框架会按照不同的顺序存储特征图数据。
- 比如Caffe，排列顺序为[Batch, Channels, Height, Width]，即NCHW。
- TensorFlow中，排列顺序为[Batch, Height, Width, Channels]，即NHWC。

<img src="./images/data_layout_format_example.png" alt="data_layout_format_example"  width="500px" >

分别按照NHWC与NCHW的顺序分别对下图进行遍历，注意遍历是按照**从低维到高维**进行的。

<img src="./images/rgb_image_example.png" alt="rgb_image_example"  width="500px" >

图中几个维度大小分别为N：1，H：3，W：3，C：3，不同的排布格式遍历顺序如下所示：
- NHWC：R00,G00,B00,R01,G01,B01,R02,G02,B02,R10,G10,B10,R11,G11,B11,R12,……
- NCHW：R00,R01,R02,R10,R11,R12,R20,R21,R22,G00,G01,G02;G10,G11,G12,G20,……

In [ ]:
import numpy as np

# 1. 创建模拟图片：N=1, H=3, W=3, C=3（对应教程中的RGB图片举例）
# 初始为NHWC格式（TensorFlow默认）：shape=(1,3,3,3)
img_nhwc = np.array([
    [
        [[255,0,0], [255,255,0], [0,255,0]],
        [[0,255,255], [0,0,255], [255,0,255]],
        [[128,128,128], [0,0,0], [255,255,255]]
    ]
], dtype=np.uint8)
print("【NHWC格式】shape =", img_nhwc.shape)
print("NHWC遍历（低维到高维：N→H→W→C）：\n", img_nhwc.flatten()[:12])  # 展平看前12个元素

# 2. NHWC转换为NCHW格式（Caffe/PyTorch默认）：用transpose重排维度
# transpose(0,3,1,2)：将维度顺序从(N,H,W,C)改为(N,C,H,W)
img_nchw = img_nhwc.transpose(0, 3, 1, 2)
print("\n【NCHW格式】shape =", img_nchw.shape)
print("NCHW遍历（低维到高维：N→C→H→W）：\n", img_nchw.flatten()[:12])  # 展平看前12个元素

# 3. 格式还原：NCHW→NHWC
img_nhwc_back = img_nchw.transpose(0, 2, 3, 1)
print("\n【NCHW转回NHWC】是否与原数据一致：", np.array_equal(img_nhwc, img_nhwc_back))

根据代码，尝试以下操作理解format。
- 运行代码，看NHWC和NCHW的展平遍历顺序。
- 查看img_nhwc.flatten()和img_nchw.flatten()的完整结果，对比两种格式的线性存储差异。

#### 2.2.6 轴（Axis）
除了以上概念外，tensor中还有一个比较常用的概念，叫做轴。它代表张量中维度的下标。比如张量a是一个5行6列的二维数组，即shape是(5,6)，则axis=0表示是张量中的第一维，即行。axis=1表示是张量中的第二维，即列。

例如张量数据：[[[1,2],[3,4]], [[5,6],[7,8]]]，Shape为(2,2,2)
- 轴0代表第一个维度的数据：[[1,2],[3,4]]与[[5,6],[7,8]]这两个矩阵
- 轴1代表第二个维度的数据即[1,2]、[3,4]、[5,6]、[7,8]这四个数组
- 轴2代表第三个维度的数据即1，2，3，4，5，6，7，8这八个数。

轴axis可以为负数，此时表示是倒数第axis个维度。N维Tensor的轴有：0 , 1, 2,……，N-1。

In [ ]:
import numpy as np

# 1. 创建教程中的3维Tensor：shape=(2,2,2)
tensor = np.array([[[1,2],[3,4]], [[5,6],[7,8]]])
print("【原始3维Tensor】shape=(2,2,2)\n", tensor)
print("-"*30)

# 2. 按不同轴提取数据，理解axis的含义
print("【按轴提取数据】")
print("axis=0（第一维）：\n", tensor[0], "\n", tensor[1])  # 轴0的两个元素
print("axis=1（第二维）：\n", tensor[:,0,:], "\n", tensor[:,1,:])  # 轴1的所有元素
print("axis=2（第三维）：\n", tensor[:,:,0], "\n", tensor[:,:,1])  # 轴2的所有元素
print("-"*30)

# 3. 体验负轴：倒数第n个维度
print("【负轴体验】")
print("axis=-1（倒数第一维，即axis=2）：", tensor[:,:,-1])
print("axis=-2（倒数第二维，即axis=1）：", tensor[:,-1,:])
print("-"*30)

# 4. 轴在算子计算中的作用：按指定轴求和（求和算子）
sum_op = lambda x, axis: np.sum(x, axis=axis)  # 求和算子，指定轴
print("【求和算子（按不同轴计算）】")
print("按axis=0求和：", sum_op(tensor, axis=0), "，shape：", sum_op(tensor, axis=0).shape)
print("按axis=1求和：", sum_op(tensor, axis=1), "，shape：", sum_op(tensor, axis=1).shape)
print("按axis=2求和：", sum_op(tensor, axis=2), "，shape：", sum_op(tensor, axis=2).shape)
print("按axis=None求和（所有元素）：", sum_op(tensor, axis=None))

根据代码，尝试以下操作理解轴的概念。
- 运行代码，看不同axis对应的维度数据，理解 “轴 = 维度下标” 的核心；
- 对4D的图片Tensor（shape=(4,20,20,3)）按axis=3（通道）求和，理解其物理含义（像素灰度化）。

---

## 课后练习
本节总体介绍了AI的发展历程、算子的含义及基本核心概念，请根据学习内容完成以下题目进行自测。

1. （判断题）算子类型决定了算子的核心实现逻辑；同一类型的算子在计算逻辑上应一致。

2. （判断题）在NCHW格式中，通道（Channel）位于axis=1。

3. （单选题）以下哪一项不是“算子类型”的例子，而是“数据容器”？  
    A. Tensor   
    B. ReLU  
    C. Conv2D  
    D. Pooling

4. （单选题）将 shape=[1,3,224,224] 从NCHW转换为NHWC，得到的新shape是？  
    A. [3,1,224,224]  
    B. [224,224,1,3]  
    C. [1,3,224,224]  
    D. [1,224,224,3]

5. （多选题）关于TensorDesc、shape、format、Axis，下列说法正确的是：  
    A. shape 定义了每个轴上的元素数量  
    B. format 决定了轴的语义顺序（如NCHW vs NHWC）  
    C. Axis 只能是正整数  
    D. TensorDesc 可能包含数据类型信息  

**执行以下代码获取答案。**


In [ ]:
!cat ./answer/01.02_answer.txt